# Tutorial #1 Simulation with NeuroMTA IP

In this tutorial, we are going to simulate performance of Tenstorrent-like multi-tile accelerator architecture by using NeuroMTA IP.

### STEP 1: Import necessary libraries

In [ ]:
import torch

from neuromta.framework import *        # NeuroMTA Framework
from neuromta.hardware import *         # NeuroMTA Hardware
from neuromta.ip.tenstorrent import *   # NeuroMTA IP for Tenstorrent-like architecture

### STEP 2: Create device instance

In [ ]:
config = TenstorrentConfig.BLACKHOLE()
device = TenstorrentDevice(**config)    # Create device instance
device.initialize()

device.print_summary()

### STEP 3: Create workload (Linear + ReLU)

In [ ]:
M = 512
N = 512
K = 512
dtype = torch.int8
acc_dtype = torch.int32

ifm:  torch.Tensor = torch.randint(-32, 32, (M * K,)).to(dtype=dtype).reshape(M, K)
wgt:  torch.Tensor = torch.randint(-32, 32, (K * N,)).to(dtype=dtype).reshape(K, N).T  # (N, K)
bias: torch.Tensor = torch.randint(-32, 32, (N,)).to(dtype=acc_dtype).flatten()

### STEP 4: Define memory layout and load data to the device memory

In this step, you need to define the memory layout of each tensor. The `torch` tensor has its own shape and stored with row-major layout. However, `NPUCore` of the `TenstorrentDevice` instance requires the tensor to be tiled into $32\times32$ fixed-size pages. To fully utilize the off-chip device memory bandwidth, pages must be interleaved to the memory channels to make sure that the pages can be accessed in parallel by multiple cores.

You can simply create the paged tensor buffer by using `MCA_TensorBuffer`. This function receives `MCA_TensorMemoryLayout`, which determines the layout and the memory type of the buffer.

In [ ]:
# Define cores to be used for this workload
core_grid = device.get_npu_core_grid(offset=(0, 0), shape=(4, 4))
core_ids = core_grid.core_ids

# Define memory layout for MAIN and L1 buffers
main_layout = MCA_TensorMemoryLayout(mem_type=MCA_TensorMemoryType.MAIN, page_shape=(32, 32))
l1_layout = MCA_TensorMemoryLayout(mem_type=MCA_TensorMemoryType.L1, page_shape=(32, 32))    

# Create MAIN tensor buffers
main_buf_ifm  = MCA_TensorBuffer(shape=ifm.shape,  dtype=ifm.dtype,  layout=main_layout, device=device)
main_buf_wgt  = MCA_TensorBuffer(shape=wgt.shape,  dtype=wgt.dtype,  layout=main_layout, device=device)
main_buf_psum = MCA_TensorBuffer(shape=bias.shape, dtype=bias.dtype, layout=main_layout.overrides(page_shape=(1, 32)), device=device)
main_buf_ofm  = MCA_TensorBuffer(shape=(M, N), dtype=acc_dtype, layout=main_layout, device=device)

# Create L1 tensor buffers
l1_buf_ifm  = MCA_TensorBuffer(shape=ifm.shape,  dtype=ifm.dtype,  layout=l1_layout, device=device, core_ids=core_ids)
l1_buf_wgt  = MCA_TensorBuffer(shape=wgt.shape,  dtype=wgt.dtype,  layout=l1_layout, device=device, core_ids=core_ids)
l1_buf_psum = MCA_TensorBuffer(shape=bias.shape, dtype=bias.dtype, layout=l1_layout.overrides(page_shape=(1, 32)), device=device, core_ids=core_ids)
l1_buf_ofm  = MCA_TensorBuffer(shape=(M, N), dtype=acc_dtype, layout=l1_layout, device=device, core_ids=core_ids)

# Load data to the MAIN memory buffers
main_buf_ifm.update(ifm)
main_buf_wgt.update(wgt)
main_buf_psum.update(bias)

### STEP 5: Call runtime operators

In this step, you are going to call predefined operators. The code below loads tensors to the L1 memory, and executes `linear` and `relu` operators. The output feature map will be stored back to the main memory buffer.

Note that each operator is not executed right after it is called. All the operators are just-in-time compiled and dispatched to the designated cores. 

In [ ]:
MCA_RT_DMA_LOAD(device, main_buf_ifm, l1_buf_ifm)
MCA_RT_DMA_LOAD(device, main_buf_wgt, l1_buf_wgt)
MCA_RT_DMA_LOAD(device, main_buf_psum, l1_buf_psum)

MCA_RT_GLOBAL_SYNC(device, core_grid.core_ids)

TT_RT_LINEAR(
    device=device, core_grid=core_grid,
    buf_ifm=l1_buf_ifm, buf_wgt=l1_buf_wgt, buf_bias=l1_buf_psum, buf_ofm=l1_buf_ofm,
    dtype=dtype, acc_dtype=acc_dtype,
)

MCA_RT_GLOBAL_SYNC(device, core_grid.core_ids)

TT_RT_RELU(device, core_grid, l1_buf_ofm, inplace=True)  # TODO: is this layer fusion??

MCA_RT_GLOBAL_SYNC(device, core_grid.core_ids)

MCA_RT_DMA_STORE(device, l1_buf_ofm, main_buf_ofm)

Finally, run all the kernels by using `run_kernels()` method.

In [ ]:
logger.set_print_options(LogLevel.DEBUG)  # Set logger to print debug messages
device.set_command_debug_verbosity()      # Set device to print debug messages for each command

device.run_kernels()  # Start executing the enqueued commands

In [ ]:
# Verify the result with PyTorch
reference = torch.relu(torch.nn.functional.linear(ifm.to(acc_dtype), wgt.to(acc_dtype), bias.to(acc_dtype)))
simulated = main_buf_ofm.restore()

print(f"\n=== REFERENCE ===\n{reference}")
print(f"\n=== SIMULATED ===\n{simulated}")
print(f"\nnumber of mismatched elements: {torch.sum(reference != simulated)} / {torch.numel(reference)}")
print(f"simulation terminated with valid result: {torch.allclose(reference, simulated)}")